# SI Table 1: global mutation-rate fits with 95% bootstrap confidence intervals

Intervals are plain percentiles over B = 10,000 barcode-level resamples.  Each
replicate redoes the entire downstream analysis, not just the final optimisation: the
binning is recomputed, because it depends on the resampled counts and reusing the
original binning would understate the variability.

Per-base rates divide by 236 bp, the effective canavanine loss-of-function target
size, so they can be compared with published per-base mutation rates.

Reads the stored bootstrap replicates in `data/bootstrap/` rather than regenerating them.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "barfly").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from IPython.display import Markdown, display
import numpy as np

OUT = ROOT / "output"; OUT.mkdir(exist_ok=True)
BOOT = ROOT / "data" / "bootstrap"
TARGET_BP = 236          # canavanine loss-of-function target size

STRAIN_LABEL = {"yKH88": "yKH88 (BY4741)",
                "yFR1": "yFR1 (BY4741 *msh6Δ*)",
                "yFR2": "yFR2 (BY4741 *msh2Δ*)"}
METHODS = ["least squares", "Laplace"]

In [2]:
def percentile_interval(point, samples):
    lower, upper = np.percentile(samples, [2.5, 97.5])
    return point, lower, upper

In [3]:
def sig(x, n=3):
    """Round to n significant figures, printed without trailing noise."""
    if x == 0 or not np.isfinite(x):
        return "-"
    from math import floor, log10
    d = n - 1 - int(floor(log10(abs(x))))
    v = round(x, d)
    return f"{v:g}"

In [4]:
sys.path.insert(0, str(ROOT))

OUT = ROOT / "output"; OUT.mkdir(exist_ok=True)
BOOT = ROOT / "data" / "bootstrap"
TARGET_BP = 236          # canavanine loss-of-function target size

STRAIN_LABEL = {"yKH88": "yKH88 (BY4741)",
                "yFR1": "yFR1 (BY4741 *msh6Δ*)",
                "yFR2": "yFR2 (BY4741 *msh2Δ*)"}
METHODS = ["least squares", "Laplace"]


def percentile_interval(point, samples):
    lower, upper = np.percentile(samples, [2.5, 97.5])
    return point, lower, upper


def sig(x, n=3):
    """Round to n significant figures, printed without trailing noise."""
    if x == 0 or not np.isfinite(x):
        return "-"
    from math import floor, log10
    d = n - 1 - int(floor(log10(abs(x))))
    v = round(x, d)
    return f"{v:g}"


rows = []

## Expt 1: point/boot axes (method, k-row, parameter); k-row -1 = global

In [5]:
expt1 = np.load(BOOT / "20260801_expt1_bootstrap.npz")
for method, name in enumerate(METHODS):
    point, lower, upper = percentile_interval(expt1["point"][method, -1, 0], expt1["boot"][:, method, -1, 0])
    rows.append(("Fig. 2 — BY4741 *msh2Δ*", name, point, lower, upper))

## Expt 2: axes (strain, method, k-row, parameter)

In [6]:
expt2 = np.load(BOOT / "20260801_expt2_bootstrap.npz")
for i, strain in enumerate(expt2["strains"]):
    for method, name in enumerate(METHODS):
        point, lower, upper = percentile_interval(expt2["point"][i, method, -1, 0], expt2["boot"][:, i, method, -1, 0])
        rows.append((f"Fig. 3 — {STRAIN_LABEL[str(strain)]}", name, point, lower, upper))

lines = [
    "# SI Table 1 — global mutation-rate fits with 95% bootstrap CIs",
    "",
    f"Global (all-*k*, summed-loss) fits. B = {int(expt1['B'])} barcode-level resamples; "
    "plain percentile intervals (2.5th and 97.5th).",
    f"Per-base columns divide by {TARGET_BP} bp, the effective canavanine target size.",
    "",
    "| dataset | fit | μ (×10⁻⁶) | 95% CI (×10⁻⁶) | μ per base (×10⁻⁹) | 95% CI per base (×10⁻⁹) |",
    "|---|---|---|---|---|---|",
]
prev = None
for dataset, name, point, lower, upper in rows:
    label = "" if dataset == prev else dataset
    prev = dataset
    lines.append(
        f"| {label} | {name} | {sig(point*1e6)} | "
        f"[{sig(lower*1e6)}, {sig(upper*1e6)}] | "
        f"{sig(point/TARGET_BP*1e9)} | "
        f"[{sig(lower/TARGET_BP*1e9)}, {sig(upper/TARGET_BP*1e9)}] |")

In [7]:
(OUT / "table01_bootstrap_cis.md").write_text("\n".join(lines) + "\n")
display(Markdown("\n".join(lines)))

# SI Table 1 — global mutation-rate fits with 95% bootstrap CIs

Global (all-*k*, summed-loss) fits. B = 10000 barcode-level resamples; plain percentile intervals (2.5th and 97.5th).
Per-base columns divide by 236 bp, the effective canavanine target size.

| dataset | fit | μ (×10⁻⁶) | 95% CI (×10⁻⁶) | μ per base (×10⁻⁹) | 95% CI per base (×10⁻⁹) |
|---|---|---|---|---|---|
| Fig. 2 — BY4741 *msh2Δ* | least squares | 2.04 | [1.51, 2.89] | 8.64 | [6.4, 12.3] |
|  | Laplace | 2.12 | [1.69, 2.37] | 8.99 | [7.14, 10] |
| Fig. 3 — yKH88 (BY4741) | least squares | 0.285 | [0.248, 0.369] | 1.21 | [1.05, 1.56] |
|  | Laplace | 0.433 | [0.348, 0.518] | 1.83 | [1.47, 2.19] |
| Fig. 3 — yFR1 (BY4741 *msh6Δ*) | least squares | 0.979 | [0.86, 1.44] | 4.15 | [3.65, 6.12] |
|  | Laplace | 1.15 | [0.904, 1.39] | 4.86 | [3.83, 5.88] |
| Fig. 3 — yFR2 (BY4741 *msh2Δ*) | least squares | 2.87 | [1.91, 9.95] | 12.1 | [8.08, 42.1] |
|  | Laplace | 2.04 | [1.81, 2.3] | 8.66 | [7.69, 9.76] |